<a href="https://colab.research.google.com/github/William-abing/ConsolePi/blob/master/examples/colab/funasr_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FunASR Colab Quickstart

Run FunASR in a browser, transcribe a public sample, then try your own audio file.

Repository: https://github.com/modelscope/FunASR


## 1. Install dependencies

The first cell can take a few minutes because it installs FunASR and downloads Python wheels. Colab already includes PyTorch in most runtimes.


In [3]:
!pip -q install -U funasr modelscope soundfile

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 20.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.9/925.9 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.5/99.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 83.8 MB/s eta 0:00:00


## 2. Select CPU or GPU

For a faster run, choose **Runtime -> Change runtime type -> GPU** in Colab before running the notebook.


In [4]:
import json
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## 3. Transcribe a public sample

This uses `paraformer-zh` with VAD and punctuation so the example works with a short public Mandarin sample.


In [11]:
from funasr import AutoModel

sample_url = "https://isv-data.oss-cn-hangzhou.aliyuncs.com/ics/MaaS/ASR/test_audio/vad_example.wav"

model = AutoModel(
    model="paraformer-zh",
    vad_model="fsmn-vad",
    punc_model="ct-punc",
    device=device,
    hub="hf",
    disable_update=True,
    disable_pbar=True,
    log_level="ERROR"
)

result = model.generate(input=sample_url, batch_size_s=60)
print(json.dumps(result, ensure_ascii=False, indent=2))

funasr version: 1.3.13.


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

[
  {
    "key": "vad_example",
    "text": "试错的过程很简单，而且特别是今天报名唱学卡的同学，你们可以听到后面的有专门的活动课，他会大大降低你的试错成本。其实你也可以不要来听课，为什么你自己写嘛？我先今天写五个点，我就实试实验一下，发现这五个点不行，我再写五个点，这实在不行，那再写五个点嘛。你总会所谓的活动大神和所谓的高手，就是只有一个把所有的错，所有的坑全国趟一遍，留下正确的。你就是所谓的大神，明白吗？所以说关于活动通过这一块，我只送给你们四个字啊，换位思考。如果说你要想降低你的试错成本，今天来这里你们就是对的。因为有唱学唱学卡这个机会，所以说关于活动过于不过这个问题或者活动很难通过这个话题。呃，如果真的要坐下来聊的话，要聊一天，但是我觉得我刚才说的四个字足够好，谢谢。好，非常感谢那个三毛老师的回答啊。三毛老师说我们在整个店铺的这个活动当中，我们要学会换位思考。其实。"
  }
]


## 4. Try your own audio file

Upload a short `.wav`, `.mp3`, `.m4a`, or `.flac` file. For long recordings, split the audio or use a GPU runtime.


In [12]:
from google.colab import files

uploaded = files.upload()
audio_path = next(iter(uploaded))
print(f"Uploaded: {audio_path}")

user_result = model.generate(input=audio_path, batch_size_s=60)
print(json.dumps(user_result, ensure_ascii=False, indent=2))

Saving Recording.wav to Recording (1).wav
Uploaded: Recording (1).wav
[
  {
    "key": "Recording (1)",
    "text": " Hello hello声音测试声音测试录音测试你好你好。"
  }
]


In [14]:
import soundfile as sf
import soxr
import wave
import numpy as np
model = AutoModel(
    model="paraformer-zh-streaming",
    device="cuda",
    disable_update=True,
    hub="hf"
)

cache = {}

with wave.open(audio_path, 'rb') as wf:
  channels = wf.getnchannels()
  sampwidth = wf.getsampwidth()
  original_rate = wf.getframerate()
  total_frames = wf.getnframes()

  print(channels, sampwidth, original_rate, sep=",")
  original_chunk_samples = original_rate * 600 // 1000
  resampler = soxr.ResampleStream(
      in_rate=original_rate,
      out_rate=16000,
      num_channels=2,
      dtype="int16"
  )

  read_frames = 0
  while original_chunk := wf.readframes(original_chunk_samples):
    read_frames += (len(original_chunk) // (channels * sampwidth))
    is_final = read_frames >= total_frames
    # print(f"Read frames: {read_frames}/{total_frames}, is final: {is_final}")
    original_np_chunk = (np.frombuffer(original_chunk, dtype=np.int16)
    .reshape(-1, 2)
    # .mean(axis=1).astype(np.int16)
    )

    asr_chunk = resampler.resample_chunk(original_np_chunk)
    res = model.generate(
        input=asr_chunk[:,0],
        # input=asr_chunk,
        cache=cache,
        is_final=is_final,
        chunk_size=[0, 10, 5],
        disable_pbar=True
    )
    if res and len(res) > 0:
      print(res[0]["text"], end="|", flush=True)


print(audio_path)

audio, sr = sf.read(audio_path, dtype="float32")
chunk_size = [0, 10, 5]
chunk_stride = chunk_size[1] * 960

# cache = {}
# n_chunks = (len(audio) - 1) // chunk_stride + 1
# print(len(audio), sr, chunk_stride, sep="\n")

# for i in range(n_chunks):
#   chunk = audio[i * chunk_stride : (i + 1) * chunk_stride]
#   res = model.generate(
#       input=chunk,
#       cache=cache,
#       is_final=(i == n_chunks - 1),
#       chunk_size=chunk_size,
#       encoder_chunk_look_back=4,
#       decoder_chunk_look_back=1
#   )
#   if chunk_text := res[0]["text"]:
#     print(chunk_text, end="|", flush=True)

funasr version: 1.3.13.


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

2,2,48000
||hello||lo|声音测|试声音测|试录音|测试||你|好你|好|Recording (1).wav


In [25]:
vad_model = AutoModel(
                      # model="paraformer-zh-streaming",
                      model="fsmn-vad",
                      # model="iic/speech_fsmn_vad_zh-cn-16k-common-pytorch",
                      device="cuda",
                      disable_pbar=True,
                      hub="hf"
                      )
asr_cache = {}
vad_cache = {}
with wave.open(audio_path, 'rb') as wf:
  channels = wf.getnchannels()
  sampwidth = wf.getsampwidth()
  sample_rate = wf.getframerate()
  total_frames = wf.getnframes()

  chunk_samples = sample_rate * 200 // 1000
  resampler = soxr.ResampleStream(
      in_rate=sample_rate,
      out_rate=16000,
      num_channels=channels,
      dtype="int16"
  )

  read_frames = 0

  while chunk := wf.readframes(chunk_samples):
    read_frames += (len(chunk) // (sampwidth * channels))
    is_final = (read_frames >= total_frames)
    np_chunk = (
        np.frombuffer(chunk, dtype=np.int16)
        .reshape(-1, channels)
        # .mean(axis=1).astype(np.int16)
    )

    asr_chunk = resampler.resample_chunk(np_chunk)
    print(len(np_chunk), len(asr_chunk))
    vad_res = vad_model.generate(
        input=asr_chunk[:,0],
        cache=vad_cache,
        is_final=is_final,
        chunk_size=200,
        # key="vad-stream"
    )
    print(vad_res)


funasr version: 1.3.13.
Check update of funasr, and it would cost few times. You may disable it by set `disable_update=True` in AutoModel
You are using the latest version of funasr-1.3.13


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

9600 2936
[{'key': 'rand_key_2yW4Acq9GFz6Y', 'value': []}]
9600 2936
[{'key': 'rand_key_1t9EwL56nGisi', 'value': []}]
9600 3426
[{'key': 'rand_key_WgNZq6ITZM5jt', 'value': []}]
9600 2936
[{'key': 'rand_key_gUe52RvEJgwBu', 'value': []}]
9600 3425
[{'key': 'rand_key_NO6n9JEC3HqdZ', 'value': [[320, -1]]}]
9600 3425
[{'key': 'rand_key_6J6afU1zT0YQO', 'value': []}]
9600 2936
[{'key': 'rand_key_aNF03vpUuT3em', 'value': []}]
9600 3426
[{'key': 'rand_key_6KopZ9jZICffu', 'value': []}]
9600 2936
[{'key': 'rand_key_4G7FgtJsThJv0', 'value': []}]
9600 3425
[{'key': 'rand_key_7In9ZMJLsCfMZ', 'value': []}]
9600 2936
[{'key': 'rand_key_yuKpslm0lcNQq', 'value': []}]
9600 3425
[{'key': 'rand_key_EefRWi4j7c1f5', 'value': []}]
9600 2936
[{'key': 'rand_key_S71IRz1THrHZp', 'value': []}]
9600 3426
[{'key': 'rand_key_2n5RL08ALrCFQ', 'value': []}]
9600 2936
[{'key': 'rand_key_PS6YwfuNhFLOv', 'value': []}]
9600 3425
[{'key': 'rand_key_2mpbUrhToxYkv', 'value': []}]
9600 2936
[{'key': 'rand_key_B0dgYj2Soc0KO', 'v

## 5. Save the transcript JSON

Attach this JSON when you open a GitHub issue or compare model outputs.


In [ ]:
from pathlib import Path

Path("funasr_transcript.json").write_text(
    json.dumps(user_result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
files.download("funasr_transcript.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Next steps

- Choose a model: https://github.com/modelscope/FunASR/blob/main/docs/model_selection.md
- Compare with Whisper or cloud ASR: https://github.com/modelscope/FunASR/blob/main/docs/migration_from_whisper.md
- Deploy an OpenAI-compatible API: https://github.com/modelscope/FunASR/tree/main/examples/openai_api
- Read production deployment options: https://github.com/modelscope/FunASR/blob/main/docs/deployment_matrix.md
